# W1.2 bulk OCR v2: Bodhan IndicOCR over the scanned corpus (Colab or Kaggle)

Madhya Kanoon Saathi, DEC-001 / DEC-053 / DEC-069. Needs a **GPU (T4)**. 21 documents, 635 pages, split into packages
`mks_ocr_bulk_p1` ... `_p20` (`scripts/prepare_ocr_bulk.py --parts N`). Run one part per session.

**New in v2: resume from pages already done anywhere.** Before OCR starts, the notebook looks for earlier outputs
(an `out/` folder with `run_info.json`, or a `mks_ocr_bulk*_out*.zip`) in the mounted Drive, in folders shared with
you, or in Kaggle inputs. It also checks any paths you list in `RESUME_FROM`. Finished pages that belong to this part are copied in, so OCR starts at
the first page not yet done. A source is used only if its `run_info.json` shows the same Bodhan revision and dpi.
This lets you move a part to another Google account: upload its partial `out/` folder (or ZIP) next to the
package and run again.

**Colab**
1. Upload the unzipped `mks_ocr_bulk_pN/` folder to the Drive of the account that runs Colab (anywhere in My Drive).
   If the folder is shared from another account and the search finds nothing, set `DRIVE_DIR` to its path.
2. Secret `HF_TOKEN` (key icon), runtime **T4 GPU**, Run all. After a timeout or disconnect, Run all again.
3. The output ZIP `mks_ocr_bulk_pN_out.zip` lands in Drive root when "pages missing: 0".

**Kaggle**: private Dataset with the part ZIP -> Add Input; Accelerator **GPU T4 x2**; Internet On; secret `HF_TOKEN`;
Save & Run All. To continue, attach the previous version's output as an input and run again.

Unzip each output into `data/ocr/bulk/` in the repo. Nothing here prints secrets.

## 0. Platform, settings and GPU check (stops early on a CPU runtime)

In [ ]:
import datetime
import hashlib
import importlib.util
import json
import pathlib
import re
import shutil
import time
import zipfile

import torch

# Colab first: Colab machines also have a /kaggle/input folder, so that alone is not a signal
COLAB = importlib.util.find_spec("google.colab") is not None
KAGGLE = not COLAB and pathlib.Path("/kaggle/input").exists()
assert COLAB or KAGGLE, "run this on Google Colab or Kaggle"

# ---- settings (usually leave as they are) ----
PART = None  # only if several parts are visible at once: the part number N of mks_ocr_bulk_pN
DRIVE_DIR = None  # Colab: path to the package folder if the search does not find it
RESUME_FROM = []  # extra earlier outputs: paths of out/ folders or *_out*.zip files
MAX_HOURS = 11.0  # Kaggle stops a session at its limit (~12 h, not verified); stop before it
# same revision as the W0.5 bake-off run (data/ocr/out/run_info.json)
REVISION = "cd50d301d0e17e8ecb32fc49c8ccbd7914dcfc25"

# Bodhan needs a GPU. A CPU runtime has a CPU-only torch and fails with
# "Torch not compiled with CUDA enabled".
assert torch.cuda.is_available(), (
    "No GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2. "
    "Colab: Runtime -> Change runtime type -> T4 GPU."
)
GPU = torch.cuda.get_device_name(0)
if torch.cuda.get_device_capability(0) < (7, 0):
    print("WARNING: old GPU", GPU, "- recent torch may not support it; prefer GPU T4")
print("platform", "kaggle" if KAGGLE else "colab", "| torch", torch.__version__, "| GPU", GPU)

## 1. Inputs, outputs, sha256 check

In [ ]:
def n_done(pkg_dir):
    return sum(1 for _ in (pkg_dir / "out").glob("*/p*.md"))


def find_package(roots):
    # a package = folder holding jobs.json + pdfs/; its name comes from jobs.json, because
    # Kaggle names the folder after the dataset (e.g. mks-ocr-bulk-p1), not after the zip
    found = {}
    for root in roots:
        if not root.exists():
            continue
        for p in sorted(root.rglob("jobs.json")):
            if p.parent.name != "out" and (p.parent / "pdfs").is_dir():
                j = json.loads(p.read_text(encoding="utf-8"))
                if PART is None or j.get("part") == PART:
                    found.setdefault(j.get("package", "mks_ocr_bulk"), []).append(p.parent)
    if len(found) != 1:
        for root in roots:
            for p in sorted(root.glob("*/*"))[:30]:
                print("  seen:", p)
    assert len(found) == 1, f"expected one package, found {sorted(found)}; set PART or DRIVE_DIR"
    pkg, dirs = next(iter(found.items()))
    # the same package seen twice (e.g. My Drive + a shared shortcut): write into the copy with
    # the most finished pages; the others are picked up as resume sources below
    dirs.sort(key=n_done, reverse=True)
    if len(dirs) > 1:
        print("package found in several places, using", dirs[0], "| others:", dirs[1:])
    return dirs[0], pkg


if KAGGLE:
    # input dataset is read-only; outputs go to /kaggle/working (saved with the version)
    ROOTS = [pathlib.Path("/kaggle/input")]
    IN, PKG = find_package(ROOTS)
    BASE = pathlib.Path("/kaggle/working")
    OUT = BASE / PKG / "out"
    OUT.mkdir(parents=True, exist_ok=True)
else:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
    if DRIVE_DIR:
        ROOTS = [pathlib.Path(DRIVE_DIR)]
    else:
        # folders shared from another account show up under .shortcut-targets-by-id
        ROOTS = [BASE, pathlib.Path("/content/drive/.shortcut-targets-by-id")]
    IN, PKG = find_package(ROOTS)
    OUT = IN / "out"
    OUT.mkdir(exist_ok=True)
print("package", PKG, "from", IN)

JOBS = json.loads((IN / "jobs.json").read_text(encoding="utf-8"))
INFO_PATH = OUT / "run_info.json"
INFO = json.loads(INFO_PATH.read_text()) if INFO_PATH.exists() else {"pages": {}}
SESSION = {
    "platform": "kaggle" if KAGGLE else "colab",
    "gpu": GPU,
    "torch": torch.__version__,
    "started_at": datetime.datetime.now(datetime.UTC).isoformat(),
}
INFO.setdefault("sessions", []).append(SESSION)


def save_info():
    INFO_PATH.write_text(json.dumps(INFO, indent=1, ensure_ascii=False))


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


bad = [j["doc_id"] for j in JOBS["jobs"] if sha256(IN / j["file"]) != j["sha256"]]
assert not bad, f"sha256 mismatch (re-upload these PDFs): {bad}"
save_info()
print(len(JOBS["jobs"]), "documents,", JOBS["total_pages"], "pages; all sha256 match")

## 2. Resume: copy in pages already done elsewhere (same revision and dpi only)

In [ ]:
# pages this part must produce: output folder name -> page numbers (1-based, original numbering)
WANT = {}
for j in JOBS["jobs"]:
    pages = range(j.get("first", 1), j.get("last", j["pages"]) + 1)
    WANT.setdefault(j["doc_id"].replace("@", "_"), set()).update(pages)
PAGE_RE = re.compile(r"(?:^|/)([^/]+)/p(\d{4})\.md$")


def compatible(run_info_text):
    # only reuse pages made by the same model revision at the same dpi
    try:
        ri = json.loads(run_info_text)
    except ValueError:
        return False
    return ri.get("bodhan", {}).get("revision") == REVISION and ri.get("dpi") == JOBS["dpi"]


def take(doc, n, md_bytes, json_bytes):
    if n not in WANT.get(doc, ()):
        return 0
    dst = OUT / doc / f"p{n:04d}.md"
    if dst.exists():
        return 0
    dst.parent.mkdir(exist_ok=True)
    if json_bytes is not None:
        (OUT / doc / f"p{n:04d}.json").write_bytes(json_bytes)
    dst.write_bytes(md_bytes)  # .md last: its presence marks the page done
    return 1


def import_dir(src):
    if not compatible((src / "run_info.json").read_text(encoding="utf-8")):
        print("  skipped (other revision/dpi):", src)
        return 0
    added = 0
    for md in src.glob("*/p*.md"):
        js = md.with_suffix(".json")
        n = int(md.stem[1:])
        added += take(md.parent.name, n, md.read_bytes(), js.read_bytes() if js.exists() else None)
    return added


def import_zip(src):
    added = 0
    with zipfile.ZipFile(src) as z:
        names = set(z.namelist())
        infos = [n for n in names if n.endswith("out/run_info.json")]
        if not infos or not all(compatible(z.read(n).decode("utf-8")) for n in infos):
            print("  skipped (no run_info or other revision/dpi):", src)
            return 0
        for name in names:
            m = PAGE_RE.search(name)
            if m:
                js = name[:-3] + ".json"
                jb = z.read(js) if js in names else None
                added += take(m[1], int(m[2]), z.read(name), jb)
    return added


def same(a, b):
    try:
        return a.resolve() == b.resolve()
    except OSError:
        return False


sources = [pathlib.Path(p) for p in RESUME_FROM]
for root in ROOTS:
    if root.exists():
        sources += [p.parent for p in root.rglob("run_info.json") if p.parent.name == "out"]
        sources += sorted(root.rglob("mks_ocr_bulk*_out*.zip"))
before = sum(1 for _ in OUT.glob("*/p*.md"))
imported = 0
for src in dict.fromkeys(sources):  # keeps order, drops duplicates
    if same(src, OUT) or not src.exists():
        continue
    k = import_zip(src) if src.suffix == ".zip" else import_dir(src)
    if k:
        print(f"  imported {k} pages from {src}")
    imported += k
SESSION["imported_pages"] = imported
save_info()

todo = [
    (d, n) for d, ns in WANT.items() for n in sorted(ns) if not (OUT / d / f"p{n:04d}.md").exists()
]
total = sum(len(ns) for ns in WANT.values())
print(f"done {total - len(todo)} / {total} ({before} already here, {imported} imported)")
print(len(todo), "pages to do")
if todo:
    print("starts at", todo[0][0], "page", todo[0][1])

## 3. Renderer: same pypdfium2 version as the repo lockfile, 300 dpi

In [ ]:
!pip -q install "{JOBS['renderer']}"
import pypdfium2 as pdfium

INFO["renderer"] = JOBS["renderer"]
INFO["dpi"] = JOBS["dpi"]
save_info()

## 4. Bodhan IndicOCR, pinned to the bake-off revision
Usage follows the model card at https://huggingface.co/bodhan-ai/indic-ocr (checked 2026-11-10).

In [ ]:
from huggingface_hub import snapshot_download

if KAGGLE:
    from kaggle_secrets import UserSecretsClient

    tok = UserSecretsClient().get_secret("HF_TOKEN")
else:
    from google.colab import userdata

    tok = userdata.get("HF_TOKEN")
REPO = snapshot_download("bodhan-ai/indic-ocr", revision=REVISION, token=tok)
del tok
INFO["bodhan"] = {
    "repo": "bodhan-ai/indic-ocr",
    "revision": REVISION,
    "licence": "Indic Open Model License v1.0",
    "table_format": "markdown",
}
save_info()
print("revision", REVISION)

In [ ]:
!cd "$REPO" && bash install.sh

## 5. OCR the remaining pages (skips pages already done; stops after MAX_HOURS)

In [ ]:
import sys

sys.path.insert(0, REPO)
from indic_ocr import IndicOCR

parser = IndicOCR.from_pretrained(REPO, table_format="markdown")
TMP = pathlib.Path("/tmp/mks_page.png")
started = time.time()
stopped = False
for j in JOBS["jobs"]:
    d = OUT / j["doc_id"].replace("@", "_")
    d.mkdir(exist_ok=True)
    pdf = None
    for n in range(j.get("first", 1), j.get("last", j["pages"]) + 1):
        key = f"{j['doc_id']}#p{n}"
        if (d / f"p{n:04d}.md").exists():
            continue  # done in this or an earlier session
        if time.time() - started > MAX_HOURS * 3600:
            stopped = True
            break
        if pdf is None:
            pdf = pdfium.PdfDocument(str(IN / j["file"]))
        img = pdf[n - 1].render(scale=JOBS["dpi"] / 72).to_pil().convert("RGB")
        img.save(TMP)
        t0 = time.time()
        page = parser.parse(str(TMP))
        js = json.dumps(page, ensure_ascii=False, indent=1)
        (d / f"p{n:04d}.json").write_text(js, encoding="utf-8")
        # the .md is written last, so its presence means the page is done
        (d / f"p{n:04d}.md").write_text(page["markdown"], encoding="utf-8")
        INFO["pages"][key] = {"seconds": round(time.time() - t0, 1), "size": list(img.size)}
        save_info()
        print(key, f"{INFO['pages'][key]['seconds']}s", len(page["markdown"]), "chars")
    if pdf is not None:
        pdf.close()
    if stopped:
        print(f"stopped after {MAX_HOURS} h; run again to continue")
        break
print("session done in", round((time.time() - started) / 60, 1), "min")

## 6. Check and package

In [ ]:
missing = [
    f"{j['doc_id']}#p{n}"
    for j in JOBS["jobs"]
    for n in range(j.get("first", 1), j.get("last", j["pages"]) + 1)
    if not (OUT / j["doc_id"].replace("@", "_") / f"p{n:04d}.md").exists()
]
print("pages missing:", len(missing), missing[:10])
if not missing:
    INFO["finished_at"] = datetime.datetime.now(datetime.UTC).isoformat()
    save_info()
    shutil.copy(IN / "jobs.json", OUT / "jobs.json")
    # zip holds <package>/out/..., so parts unzip side by side in data/ocr/bulk/
    zbase = str(BASE / f"{PKG}_out")
    print(shutil.make_archive(zbase, "zip", OUT.parent.parent, f"{OUT.parent.name}/out"))